# Neural Particle Automata [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxencefaldor/cax/blob/main/examples/32_neural_particle_automata.ipynb)

A growing neural cellular automaton lives on a lattice: every cell has an address, and its neighbours are the cells next to it forever. Here the cells are *particles*. Each carries a position as well as a state, and the rule moves both, so who is next to whom is something the automaton decides as it goes.

That one change costs the convolution. With no lattice there is no stencil, so the neighbourhood is gathered instead by smoothed particle hydrodynamics: sums over whatever particles lie within a radius, weighted by a kernel that falls smoothly to zero at the edge. The smoothness is what keeps it differentiable while neighbours come and go.

Everything else is [40 - Growing NCA](40_growing_nca.ipynb) unchanged --- the same target, the same channel count, the same stochastic half-updates, the same pool.

## Installation

You will need Python 3.12 or later, and a working JAX installation. For example, you can install JAX with:

In [ ]:
%pip install -U "jax[cuda]"

Then, install CAX from PyPi:

In [ ]:
%pip install -U "cax[examples]"

## Import

In [ ]:
import time

import jax
import jax.numpy as jnp
import mediapy
import optax
from flax import nnx
from jax import Array

from cax.core import ComplexSystem
from cax.core.perceive import Particles, SPHPerceive
from cax.nn.pool import Pool
from cax.utils import clip_and_uint8, get_emoji_array, render_states, safe_norm

## Configuration

`fused` chooses how the perception is computed. Both routes return the same numbers, so it
changes nothing about what the automaton learns --- only what it costs. The array route
builds an array with an entry per pair; the fused route accumulates the same sums a tile at
a time inside a Pallas kernel and never builds one, which is faster on a GPU and is the
only route that reaches cloud sizes the first cannot fit. It is GPU-only, so the default
follows the backend and this notebook runs anywhere.

In [ ]:
seed = 0

# The perception is the expensive part, and on a GPU it can be computed by fused
# kernels instead of array operations. Same numbers, less memory traffic.
fused = jax.default_backend() == "gpu"

num_particles = 1024
channel_size = 16
hidden_size = 128
support_radius = 0.2
seed_radius = 0.1
cell_dropout_rate = 0.5
displacement_scale = 0.5

num_steps = 96
min_steps = 64
num_eval_steps = 256
pool_size = 512
batch_size = 8
learning_rate = 5e-4
overflow_weight = 100.0

num_train_steps = 4_000
grid_size = 96
density_weight = 1.0
colour_weight = 5.0

emoji = "🦎"
size = 64

key = jax.random.key(seed)
rngs = nnx.Rngs(seed)

## Dataset

The target is a picture, but the automaton is a cloud of points, so the two are compared by drawing the cloud: each particle is splatted onto a grid, and the resulting density and colour are matched against the emoji.

Comparing per particle instead --- asking each one to match the target where it happens to stand --- would let a particle sitting on empty background simply learn to be transparent. Nothing would ever make the cloud take the shape.

In [ ]:
y = get_emoji_array(emoji, size, 0)
y = y.at[..., :3].mul(y[..., 3:])

target_density = jax.image.resize(y[..., 3:], (grid_size, grid_size, 1), "linear")
target_density = target_density / jnp.mean(target_density)
target_colour = jax.image.resize(y[..., :3], (grid_size, grid_size, 3), "linear")

mediapy.show_image(y[..., :3])

### Drawing the cloud

Twice, for two purposes. `draw` rasterizes onto a coarse grid for the loss: cheap, and smooth enough that a gradient reaches each particle's position. `render` draws a proper picture.

In [ ]:
def blur(image: Array) -> Array:
    """Blur with a separable binomial kernel."""
    weights = jnp.array([1.0, 4.0, 6.0, 4.0, 1.0]) / 16.0
    for axis in (0, 1):
        pad = [(0, 0)] * image.ndim
        pad[axis] = (2, 2)
        padded = jnp.pad(image, pad, mode="wrap")
        taps = jnp.stack(
            [
                jax.lax.dynamic_slice_in_dim(padded, i, image.shape[axis], axis)
                for i in range(weights.size)
            ]
        )
        image = jnp.tensordot(weights, taps, axes=(0, 0))
    return image


def splat(position: Array, value: Array) -> Array:
    """Rasterize particles onto a grid, each spread over the pixels it falls between."""
    pixel = position * grid_size
    corner = jnp.floor(pixel).astype(jnp.int32)
    frac = pixel - corner

    image = jnp.zeros((grid_size, grid_size, value.shape[-1]))
    for row_offset in (0, 1):
        for column_offset in (0, 1):
            weight = (frac[:, 0] if row_offset else 1.0 - frac[:, 0]) * (
                frac[:, 1] if column_offset else 1.0 - frac[:, 1]
            )
            row = (corner[:, 0] + row_offset) % grid_size
            column = (corner[:, 1] + column_offset) % grid_size
            image = image.at[row, column].add(weight[:, None] * value)
    return blur(image)


def render(particles: Particles, *, image_size: int = 384, sigma: float = 2.5) -> Array:
    """Draw the cloud for viewing: a gaussian per particle, composited over white.

    Separate from `draw` above, which the loss uses. The loss wants a cheap grid and is
    content with colour left premultiplied by density; a picture wants neither. Shown
    premultiplied, every region thinner than average comes out darkened, which reads as
    a pale and sparse cloud rather than the one that is actually there.
    """
    pixel = particles.position * image_size
    corner = jnp.floor(pixel).astype(jnp.int32)
    offset = jnp.arange(-int(3 * sigma), int(3 * sigma) + 1)
    rows, columns = jnp.meshgrid(offset, offset, indexing="ij")

    distance_row = rows[None] - (pixel - corner)[:, 0, None, None]
    distance_column = columns[None] - (pixel - corner)[:, 1, None, None]
    stamp = jnp.exp(-(distance_row**2 + distance_column**2) / (2.0 * sigma**2))
    stamp = stamp / (1e-8 + jnp.sum(stamp, axis=(1, 2), keepdims=True))

    row = (corner[:, 0, None, None] + rows[None]) % image_size
    column = (corner[:, 1, None, None] + columns[None]) % image_size
    colour = jnp.clip(particles.state[..., -4:-1], 0.0, 1.0)

    weight = (
        jnp.zeros((image_size, image_size, 1)).at[row, column].add(stamp[..., None])
    )
    painted = (
        jnp.zeros((image_size, image_size, 3))
        .at[row, column]
        .add(stamp[..., None] * colour[:, None, None, :])
    )

    alpha = jnp.clip(weight / jnp.mean(weight, where=weight > 0.0), 0.0, 1.0)
    return jnp.clip(
        (1.0 - alpha) + alpha * painted / jnp.maximum(weight, 1e-8), 0.0, 1.0
    )


def draw(particles: Particles) -> tuple[Array, Array]:
    """Draw the cloud as a density and a colour picture.

    Both are divided by the same constant, so they stay on one scale. Colour is left
    premultiplied by density rather than divided by it: dividing gives a meaningless
    colour wherever the cloud is empty.
    """
    density = splat(particles.position, jnp.ones_like(particles.state[:, :1]))
    colour = splat(particles.position, jnp.clip(particles.state[..., -4:-1], 0.0, 1.0))

    scale = jnp.mean(density)
    return density / scale, colour / scale

## Instantiate system

In [ ]:
class ParticleUpdate(nnx.Module):
    """Predict a move and a state change for every particle, from its perception."""

    def __init__(self, *, perception_size: int, rngs: nnx.Rngs):
        """Initialize particle update.

        Args:
                perception_size: Width of the perception.
                rngs: rng key.

        """
        self.layer_1 = nnx.Linear(perception_size, hidden_size, rngs=rngs)
        self.layer_2 = nnx.Linear(
            hidden_size,
            channel_size + 2,
            use_bias=False,
            kernel_init=nnx.initializers.zeros_init(),
            rngs=rngs,
        )
        self.rngs = rngs

    def __call__(
        self, state: Particles, perception: Array, input: Array | None = None
    ) -> Particles:
        """Process the current state, perception, and input to produce a new state.

        Args:
                state: Current particles.
                perception: Current perception.
                input: Optional input.

        Returns:
                Next particles.

        """
        delta = self.layer_2(nnx.relu(self.layer_1(perception)))
        move, change = delta[:, :2], delta[:, 2:]

        # Bound the move to a fraction of the support radius. Unbounded, a particle can
        # leave its neighbourhood in a single step, and everything it perceived is gone.
        # safe_norm, because the last layer starts at zero and an ordinary norm has no
        # derivative at the origin.
        length = safe_norm(move, axis=-1, keepdims=True)
        move = displacement_scale * support_radius * move / (1.0 + length)

        alive = jax.random.uniform(self.rngs.dropout(), (state.position.shape[0], 1))
        mask = (alive < cell_dropout_rate).astype(jnp.float32)

        return Particles(
            position=(state.position + mask * move) % 1.0,
            state=state.state + mask * change,
        )

In [ ]:
class NeuralParticleAutomata(ComplexSystem):
    """Neural Particle Automata class."""

    remat = True

    def __init__(self, *, rngs: nnx.Rngs):
        """Initialize Neural Particle Automata.

        Args:
                rngs: rng key.

        """
        self.perceive = SPHPerceive(
            support_radius=support_radius,
            mass=1.0 / num_particles,
            period=1.0,
            fused=fused,
        )
        self.update = ParticleUpdate(
            perception_size=SPHPerceive.perception_size(
                channel_size=channel_size, num_spatial_dims=2
            ),
            rngs=rngs,
        )

    def _step(self, state: Particles, input: Array | None = None) -> Particles:
        perception = self.perceive(state)
        next_state = self.update(state, perception, input)

        return next_state

    @nnx.jit
    def render(self, state):
        """Render state to RGB."""
        return clip_and_uint8(render(state))

In [ ]:
cs = NeuralParticleAutomata(rngs=rngs)

In [ ]:
params = nnx.state(cs, nnx.Param)
print("Number of params:", sum(x.size for x in jax.tree.leaves(params)))

## Sample initial state

Every particle starts inside a small disc, and every state starts at zero. The cloud has to carry itself out into the shape, which is the whole of what makes this a particle model rather than a lattice one wearing different clothes.

It is worth asking where the first non-zero update comes from, since a blank state through a zero-initialized output layer gives nothing back. It comes from the density gradient: the states are uniform but the positions are not, so it is the one thing a featureless cloud still has to say.

In [ ]:
def sample_state(key: Array) -> Particles:
    """Sample particles inside a disc, with no state at all."""
    angle_key, radius_key = jax.random.split(key)

    angle = jax.random.uniform(angle_key, (num_particles,), maxval=2.0 * jnp.pi)
    # The square root spreads the particles evenly over the area rather than crowding
    # them into the middle.
    radius = seed_radius * jnp.sqrt(jax.random.uniform(radius_key, (num_particles,)))
    position = 0.5 + jnp.stack(
        [radius * jnp.cos(angle), radius * jnp.sin(angle)], axis=-1
    )

    return Particles(position=position, state=jnp.zeros((num_particles, channel_size)))

## Train

### Pool

In [ ]:
key, subkey = jax.random.split(key)
particles = jax.vmap(sample_state)(jax.random.split(subkey, pool_size))

pool = Pool.create({"particles": particles})

### Optimizer

In [ ]:
lr_sched = optax.piecewise_constant_schedule(
    init_value=learning_rate, boundaries_and_scales={2_000: 0.3, 4_000: 0.3}
)

optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate=lr_sched, weight_decay=0.0),
)

optimizer = nnx.Optimizer(cs, optimizer, wrt=nnx.Param)

### Loss

Two terms match the drawn cloud to the target, and a third keeps the state in range. Nothing else bounds it, and an unbounded state is what diverges.

In [ ]:
def render_loss(particles: Particles) -> Array:
    """Squared error between the drawn cloud and the target."""
    density, colour = draw(particles)

    loss = density_weight * jnp.mean(jnp.square(density - target_density))
    return loss + colour_weight * jnp.mean(
        jnp.square(colour - target_colour * target_density)
    )


def overflow_loss(state: Array) -> Array:
    """Penalize values outside [-1, 1], over the whole rollout."""
    overflow = state - jnp.clip(state, -1.0, 1.0)
    return jnp.mean(jnp.square(overflow))

In [ ]:
def loss_fn(cs, particles, key):
    """Loss function."""
    state_axes = nnx.StateAxes({nnx.RngState: 0, ...: None})
    _, trajectory = nnx.split_rngs(splits=batch_size)(
        nnx.vmap(
            lambda cs, particles: cs(
                particles, num_steps=num_steps, return_states=True
            ),
            in_axes=(state_axes, 0),
        )
    )(cs, particles)

    # Sample a random step, so the shape has to be right over a span of time
    idx = jax.random.randint(key, (batch_size,), min_steps, num_steps)
    particles = jax.tree.map(
        lambda steps: steps[jnp.arange(batch_size), idx], trajectory
    )

    loss = jnp.mean(jax.vmap(render_loss)(particles))
    loss += overflow_weight * overflow_loss(trajectory.state)

    return loss, particles

### Train step

In [ ]:
@nnx.jit
def train_step(cs, optimizer, pool, key):
    """Train step."""
    sample_key, seed_key, loss_key = jax.random.split(key, 3)

    # Sample from pool
    pool_idx, batch = pool.sample(sample_key, batch_size=batch_size, replace=False)
    particles = batch["particles"]

    # Replace the first sample with a fresh seed
    new_particles = sample_state(seed_key)
    particles = jax.tree.map(
        lambda batch, seed: batch.at[0].set(seed), particles, new_particles
    )

    (loss, particles), grad = nnx.value_and_grad(loss_fn, has_aux=True)(
        cs, particles, loss_key
    )
    optimizer.update(cs, grad)

    pool = pool.update(pool_idx, {"particles": particles})
    return loss, pool

### Main loop

In [ ]:
print_interval = 100

losses = []
start = time.perf_counter()
for i in range(num_train_steps):
    key, subkey = jax.random.split(key)
    loss, pool = train_step(cs, optimizer, pool, subkey)
    losses.append(loss)

    if i % print_interval == 0 or i == num_train_steps - 1:
        avg_loss = sum(losses[-print_interval:]) / len(losses[-print_interval:])
        elapsed = time.perf_counter() - start
        print(f"Step {i:>4}/{num_train_steps} | {elapsed:6.1f}s | Loss {avg_loss:.3e}")

print(f"✨ Trained for {num_train_steps} steps in {time.perf_counter() - start:.0f}s")

## Run

Rolled far past the horizon it was trained on. The loss only ever looked at steps
`[min_steps, num_steps)`, so nothing asked the rule to still hold a gecko at step
`num_eval_steps` --- it either learned a shape that persists, or it learned one that
arrives on schedule and then falls apart.

In [ ]:
key, subkey = jax.random.split(key)
particles_init = sample_state(subkey)

particles_final, particles = cs(
    particles_init, num_steps=num_eval_steps, return_states=True
)

## Visualize

In [ ]:
mediapy.show_image(cs.render(particles_final))

In [ ]:
particles = jax.tree.map(
    lambda first, rest: jnp.concatenate([first[None], rest]), particles_init, particles
)
frames = render_states(cs, particles)

mediapy.show_video(frames)